# Video Games Sales Prediction

## Data Loading

In [219]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv('/workspaces/ML_01_video_game_prediction/video_game_analysis/src/data/Video_Games_Sales_as_at_22_Dec_2016.csv')

# Preview first few rows
df.head()

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


In [220]:
#Checking for missing values
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16717 non-null  object 
 1   Platform         16719 non-null  object 
 2   Year_of_Release  16450 non-null  float64
 3   Genre            16717 non-null  object 
 4   Publisher        16665 non-null  object 
 5   NA_Sales         16719 non-null  float64
 6   EU_Sales         16719 non-null  float64
 7   JP_Sales         16719 non-null  float64
 8   Other_Sales      16719 non-null  float64
 9   Global_Sales     16719 non-null  float64
 10  Critic_Score     8137 non-null   float64
 11  Critic_Count     8137 non-null   float64
 12  User_Score       10015 non-null  object 
 13  User_Count       7590 non-null   float64
 14  Developer        10096 non-null  object 
 15  Rating           9950 non-null   object 
dtypes: float64(9), object(7)
memory usage: 2.0+ MB


Name                  2
Platform              0
Year_of_Release     269
Genre                 2
Publisher            54
NA_Sales              0
EU_Sales              0
JP_Sales              0
Other_Sales           0
Global_Sales          0
Critic_Score       8582
Critic_Count       8582
User_Score         6704
User_Count         9129
Developer          6623
Rating             6769
dtype: int64

**DataSet Overview**
- 16719 entries: Total number of rows.
-  16 columns: Features in the dataset.

**Key Columns with Significant Missing Data**

- **Critic_Score**: 8,582 missing entries  
- **Critic_Count**: 8,582 missing entries  
- **User_Score**: 6,704 missing entries  
- **User_Count**: 9,129 missing entries  
- **Developer**: 6,623 missing entries  
- **Rating**: 6,769 missing entries  

**Other columns** like  *Name*,  *Platform* , *Sales* (NA, EU, JP, Other, Global),  *Genre*,  have **no missing values**, indicating relatively complete data in those areas.


## Data Cleaning

In [221]:
# Drop rows with missing target and critical fields
df = df.dropna(subset=["Global_Sales", "Year_of_Release", "Critic_Score"]) # Removing rows that have missing values for 'Global_Sales', 'Year_of_Release', or 'Critic_Score' as these fields are crucial for analysis



# Drop identifier and leakage columns
drop_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]  # Columns like sales (NA_Sales, EU_Sales, etc.) are removed because they leak information related to the target variable (Global_Sales)

for col in drop_cols:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

# The 'User_Score' column has some "tbd" values that we need to convert into NaN (or missing values) before processing it
# Convert User_Score safely (handle "tbd")
df['User_Score'] = pd.to_numeric(df['User_Score'], errors='coerce')



# We process the following numerical columns and handle missing values by filling with the median of each column
# Handle numerical columns
num_cols = ["Year_of_Release", "Critic_Score", "Critic_Count", "User_Score", "User_Count"]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")   # Convert each column to numeric, forcing errors to NaN
        df[col] = df[col].fillna(df[col].median())          # Fill missing values with the median value of the column

# Ensure counts are integers
if "User_Count" in df.columns:
    df['User_Count'] = df['User_Count'].astype(int)
if "Critic_Count" in df.columns:
    df['Critic_Count'] = df['Critic_Count'].astype(int)

# Convert Year to int
df['Year_of_Release'] = df['Year_of_Release'].astype(int)

# Handle categorical columns
# For categorical columns (e.g., Name, Platform, Genre), missing values are filled with 'Unknown'
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

- Dropped rows with missing target variables (**Global_Sales**, **Year_of_Release**, **Critic_Score**)  
- Dropped columns not useful for modeling (e.g., **NA_Sales**, **EU_Sales**, **JP_Sales**, **Other_Sales**)  
- Filled missing values:  
  - **Numerical columns** → median values  
  - **Categorical columns** → "Unknown"

## Feature Engineering

In [222]:
from collections import Counter

# Decade feature
# Creating a new feature 'Decade' based on the 'Year_of_Release'
# This classifies the games into decades (1980s, 1990s, 2000s, 2010s) for analysis
def get_decade(year):
    if year < 1990: return "1980s"
    elif year < 2000: return "1990s"
    elif year < 2010: return "2000s"
    else: return "2010s"

df["Decade"] = df["Year_of_Release"].apply(get_decade)

# Franchise Feature
# Identifying popular franchises by searching for specific keywords in the 'Name' column
franchises = ["Mario", "Pokémon", "Pokemon", "Zelda", "Call of Duty",
              "GTA", "Grand Theft Auto", "FIFA", "Final Fantasy", "Madden", "Wii Sports"]

def get_franchise(name):
    for f in franchises:
        if f.lower() in str(name).lower():
            return f
    return "Other"

df["Franchise"] = df["Name"].apply(get_franchise)

cols_to_drop = ["Name"]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)   # Dropping Name because of no use



# Log transforms of counts 
# For numerical features like 'Critic_Count' and 'User_Count', applying log transformation to reduce skewness
df["Critic_Count_Log"] = np.log1p(df["Critic_Count"])
df["User_Count_Log"] = np.log1p(df["User_Count"])

- **Decade Feature**: Categorized games into decades based on the release year  
- **Franchise Feature**: Flagged games belonging to known franchises (e.g., *Mario*, *Pokémon*)  
- **Log Transformation**: Applied to *Critic_Count* and *User_Count* to reduce skewness  
- **Normalization**: Scaled *Critic_Score* between 0 and 10 for consistency  


## Final Checks and Saving Data

In [223]:
# Checking for null values
print("After cleaning & feature engineering:", df.shape)
print("Remaining nulls:", df.isna().sum().sum())

# Save cleaned dataset
df.to_csv("vgsales_cleaned.csv", index=False)
print("Cleaned + engineered dataset saved to vgsales_cleaned.csv")

After cleaning & feature engineering: (7983, 15)
Remaining nulls: 0
Cleaned + engineered dataset saved to vgsales_cleaned.csv


- Verified the shape of the cleaned dataset  
- Ensured no missing values remained  
- Saved the cleaned dataset to **CSV** file  

## Data Split

In [224]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
import pandas as pd

# Load cleaned dataset
# Loading the cleaned dataset which has already gone through preprocessing and feature engineering
df = pd.read_csv("vgsales_cleaned.csv")

# Features & target
X = df.drop("Global_Sales", axis=1)  # Drop the target column (Global_Sales) to create feature set X
y = df["Global_Sales"]               # Target variable is 'Global_Sales'

# Stratified split based on sales bins
# We create bins for global sales to stratify the dataset and ensure that the split is representative
# Sales bins: [0, 0.1, 1, 5, 20, 100] in millions
bins = [0, 0.1, 1, 5, 20, 100]  
labels = [0, 1, 2, 3, 4]  # Labels for each bin
y_binned = pd.cut(y, bins=bins, labels=labels, include_lowest=True)  # Create binned target values

# Stratified Train-Test Split (70% - 30%)
# Splitting the data into training (70%) and temporary (30%) sets while ensuring stratification by the binned target
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y_binned
)

# Stratified Validation-Test Split (50% each of temp set)
# Now splitting the remaining 30% into validation (15%) and test (15%) sets
y_temp_binned = pd.cut(y_temp, bins=bins, labels=labels, include_lowest=True)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp_binned
)


# Save raw splits (before encoding)

# Saving the raw splits (unencoded) for future reference or reprocessing
X_train.to_csv("dataSplits/X_train.csv", index=False)
X_val.to_csv("dataSplits/X_val.csv", index=False)
X_test.to_csv("dataSplits/X_test.csv", index=False)
y_train.to_csv("dataSplits/y_train.csv", index=False)
y_val.to_csv("dataSplits/y_val.csv", index=False)
y_test.to_csv("dataSplits/y_test.csv", index=False)

print("Stratified splits saved")

Stratified splits saved


**Stratified Split**
- The dataset is split into **training**, **validation**, and **test** sets using **stratified sampling**.  
- This ensures that the distribution of **sales bins** in the target (*Global_Sales*) is preserved across all sets.  
- It ensures the model sees all types of data evenly during **training**, **validation**, and **testing**.  

**Binning Target Variable**
- The Global_Sales values are binned into categories (e.g., 0–0.1 million, 0.1–1 million, etc.) to make the target variable more suitable for classification tasks (since sales can have a large range).

## Encoding

In [225]:
#Encoding Categorical Features

# Initializing a dictionary to store LabelEncoders for each categorical column
encoders = {}

# Label Encoding for categorical columns with many unique values (Publisher, Developer, Franchise)
for col in ["Publisher", "Developer", "Franchise"]:
    if col in X_train.columns:
        le = LabelEncoder()

        # Fit the encoder on the combined unique values from train, val, and test datasets
        le.fit(list(X_train[col].unique()) +
               list(X_val[col].unique()) +
               list(X_test[col].unique()))

        # Store the encoder for later use
        encoders[col] = le  

        # Transform all datasets using the fitted encoder
        X_train[col] = le.transform(X_train[col])
        X_val[col] = le.transform(X_val[col])
        X_test[col] = le.transform(X_test[col])

# One-Hot Encoding for categorical columns with fewer unique values (Genre, Platform, Rating, Decade)
one_hot_cols = [c for c in ["Genre", "Platform", "Rating", "Decade"] if c in X_train.columns]

# One-hot encoding: Creates new binary columns for each category in the categorical features
X_train = pd.get_dummies(X_train, columns=one_hot_cols, drop_first=False)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, drop_first=False)
X_test = pd.get_dummies(X_test, columns=one_hot_cols, drop_first=False)

# Align columns across all datasets to ensure the same features
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


# Save encoded splits and encoders

# Saving the encoded datasets for future use or model training
X_train.to_csv("dataSplits/X_train_encoded.csv", index=False)
X_val.to_csv("dataSplits/X_val_encoded.csv", index=False)
X_test.to_csv("dataSplits/X_test_encoded.csv", index=False)

# Saving LabelEncoders to file for later use
joblib.dump(encoders, "encoders.pkl")

print("Encoded splits saved  & encoders stored in encoders.pkl")

Encoded splits saved  & encoders stored in encoders.pkl


**Label Encoding** 
- Applied to categorical variables: **Publisher**, **Developer**, **Franchise**.  
- Assigns an **integer value** to each unique category.  
- Useful when the number of categories is **not too large**.  



**One-Hot Encoding**  
- Applied to categorical variables with **fewer unique values**: **Genre**, **Platform**, **Rating**, **Decade**.  
- Converts each category into a **binary column** (1 or 0).  
- Helps models interpret categorical variables as **numerical features**.  



**Saving Preprocessed Data**  
- After splitting and encoding, datasets (**X_train**, **X_val**, **X_test**, and the corresponding **y** sets) are saved to **CSV files**.  
- Makes it easy to load and use the data later **without reprocessing**.  


**Saving Encoders**  
- **LabelEncoder** objects are saved to a file (`encoders.pkl`).  
- Ensures consistent encoding for **new/unseen data** in the future.

## Training

In [226]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Load the data

# Load the encoded feature data (X_train, X_test) and the target variable data (y_train, y_test)
X_train = pd.read_csv("dataSplits/X_train_encoded.csv")  # Feature set for training
y_train = pd.read_csv("dataSplits/y_train.csv").squeeze()  # Target variable for training (converted to Series)
X_test = pd.read_csv("dataSplits/X_test_encoded.csv")  # Feature set for testing
y_test = pd.read_csv("dataSplits/y_test.csv").squeeze()  # Target variable for testing (converted to Series)


# Log-transform the target
# Applying a log transformation to the target variable (Global Sales) to handle large values and skewness
# The log(1 + y) transformation helps to bring the values into a smaller range, making them more manageable for modeling
y_train_log = np.log1p(y_train)  # log(1 + Global_Sales) for training data
y_test_log = np.log1p(y_test)    # log(1 + Global_Sales) for testing data


# Define the XGBoost model

# Initializing the XGBoost Regressor model with custom hyperparameters.
# The model is set up to handle regression tasks.
xgb_model = xgb.XGBRegressor(
    random_state=42,          # Ensures reproducibility
    n_estimators=800,         # Number of trees to train
    learning_rate=0.05,       # Step size shrinking to prevent overfitting
    max_depth=6,              # Maximum depth of each tree
    subsample=0.8,            # Fraction of samples used per tree to avoid overfitting
    colsample_bytree=0.6,     # Fraction of features to use for each tree
    reg_lambda=1.0,           # L2 regularization term to reduce overfitting
    eval_metric="rmse",       # Root Mean Squared Error as the evaluation metric
    n_jobs=-1                 # Use all available CPU cores for training
)


# Train the model

# Fit the model on the training data. We also specify the test set for evaluation during training.
# This allows us to track performance on the test data after each boosting round.
xgb_model.fit(X_train, y_train_log, eval_set=[(X_test, y_test_log)], verbose=True)


# Evaluate on training data

# Predict on the training data and convert the predictions back to the original scale using inverse log transformation
y_train_pred_log = xgb_model.predict(X_train)
y_train_pred = np.expm1(y_train_pred_log)  # Convert from log scale back to original scale using exp(1) - 1

# Calculate evaluation metrics for the training set
mae_train = mean_absolute_error(y_train, y_train_pred)           # Mean Absolute Error
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))  # Root Mean Squared Error
r2_train = r2_score(y_train, y_train_pred)                       # R² (coefficient of determination)


# Print the training set evaluation metrics
print("Training Set Metrics")
print(f"MAE:  {mae_train:.4f}")
print(f"RMSE: {rmse_train:.4f}")
print(f"R²:   {r2_train:.4f}")


# Evaluate on test data

# Predict on the test data and apply inverse log transformation to convert predictions to the original scale
y_pred_log = xgb_model.predict(X_test)
y_pred = np.expm1(y_pred_log)  # Convert from log scale back to original scale using exp(1) - 1

# Calculate evaluation metrics for the test set
mae = mean_absolute_error(y_test, y_pred)   # Mean Absolute Error
mse = mean_squared_error(y_test, y_pred)    # Mean Squared Error
rmse = np.sqrt(mse)                         # Root Mean Squared Error
r2 = r2_score(y_test, y_pred)               # R² (coefficient of determination)

# Print the test set evaluation metrics
print("\nTesting Set Metrics")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


[0]	validation_0-rmse:0.42715
[1]	validation_0-rmse:0.41880
[2]	validation_0-rmse:0.41285
[3]	validation_0-rmse:0.40610
[4]	validation_0-rmse:0.39860
[5]	validation_0-rmse:0.39087
[6]	validation_0-rmse:0.38344
[7]	validation_0-rmse:0.37450
[8]	validation_0-rmse:0.36742
[9]	validation_0-rmse:0.36215
[10]	validation_0-rmse:0.35552
[11]	validation_0-rmse:0.35066
[12]	validation_0-rmse:0.34644
[13]	validation_0-rmse:0.34077
[14]	validation_0-rmse:0.33745
[15]	validation_0-rmse:0.33395
[16]	validation_0-rmse:0.32937
[17]	validation_0-rmse:0.32665
[18]	validation_0-rmse:0.32202
[19]	validation_0-rmse:0.31818
[20]	validation_0-rmse:0.31556
[21]	validation_0-rmse:0.31298
[22]	validation_0-rmse:0.31056
[23]	validation_0-rmse:0.30764
[24]	validation_0-rmse:0.30499
[25]	validation_0-rmse:0.30228
[26]	validation_0-rmse:0.29974
[27]	validation_0-rmse:0.29814
[28]	validation_0-rmse:0.29604
[29]	validation_0-rmse:0.29396
[30]	validation_0-rmse:0.29297
[31]	validation_0-rmse:0.29159
[32]	validation_0-

**Log-Transformation of Target** 
- Applied a **log(1 + y)** transformation to the target variable (**Global_Sales**).  
- Helps in **reducing skewness** of the data, making it easier for the model to handle the **large range of values**.  
- After predictions are made on the **log scale**, an **inverse transformation** using `exp(y) - 1` is applied to convert predictions back to the **original scale**.  

### Observations

#### Training Set Performance  
- **MAE (Mean Absolute Error):** 0.1048  
  - The model's predictions on the training set have an average absolute error of **0.1048** in the original scale of *Global Sales*.  
  - On average, the model's predictions are off by about **0.1 million units**.  

- **RMSE (Root Mean Squared Error):** 0.2533  
  - The model's RMSE on the training set is **0.2533**, suggesting that **larger errors** are penalized more heavily.  

- **R² (Coefficient of Determination):** 0.9828  
  - The R² value of **0.9828** indicates the model explains **98.28% of the variance** in *Global Sales*.  
  - This shows a **very strong performance** on the training data, suggesting the model fits the data well.  

#### Testing Set Performance  
- **MAE (Mean Absolute Error):** 0.3474  
  - On the test set, the MAE increased to **0.3474**.  
  - The average prediction error on unseen data is **higher** than on the training set, hinting at possible **overfitting**.  

- **RMSE (Root Mean Squared Error):** 0.7600  
  - The RMSE on the test set is **0.7600**, much **higher** than on the training set.  
  - This shows a **larger spread of errors** and **less consistency** on unseen data.
  - Reason is the Outliers and BlockBuster Games -  Blockbuster games act as outliers in the dataset. These outliers, which represent a small proportion of the dataset but have very high sales, can lead to large errors when the model predicts them, thereby increasing RMSE.  

- **R² (Coefficient of Determination):** 0.7462  
  - The R² value for the test set is **0.7462**, meaning the model explains **74.62% of the variance** in the test data.  
  - While performance is **good**, it is **significantly lower** than on the training set, suggesting the model may be **overfitting**.  


## Tuning

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -------------------------------
# 1. Load splits
# -------------------------------
X_train = pd.read_csv("dataSplits/X_train_encoded.csv")
X_val   = pd.read_csv("dataSplits/X_val_encoded.csv")
X_test  = pd.read_csv("dataSplits/X_test_encoded.csv")

y_train = pd.read_csv("dataSplits/y_train.csv").squeeze()
y_val   = pd.read_csv("dataSplits/y_val.csv").squeeze()
y_test  = pd.read_csv("dataSplits/y_test.csv").squeeze()

# Combine train + val
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

# -------------------------------
# 2. Apply log1p transform to targets
# -------------------------------
y_train_full_log = np.log1p(y_train_full)

# -------------------------------
# 3. Define hyperparameter distributions
# -------------------------------
param_dist = {
    "max_depth": [3, 4, 5, 6, 7],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [300, 500, 800, 1000],
    "reg_lambda": [0.5, 1, 2],
    "reg_alpha": [0, 0.5, 1]
}

xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)

# -------------------------------
# 4. Run RandomizedSearch
# -------------------------------
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=40,        # number of random combinations to try (can increase for more thorough search)
    scoring="r2",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

print("Running RandomizedSearchCV with log-transformed target...")

random_search.fit(X_train_full, y_train_full_log)

print("Best Params:", random_search.best_params_)
print("Best CV R²:", random_search.best_score_)

# -------------------------------
# 5. Evaluate best model on test set
# -------------------------------
best_model = random_search.best_estimator_

y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)  # back-transform

print("\nTest Metrics (Best RandomizedSearch Model with Log Transform)")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))


Running RandomizedSearchCV with log-transformed target...
Fitting 3 folds for each of 40 candidates, totalling 120 fits
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=7, min_child_weight=7, n_estimators=800, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   6.1s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=7, min_child_weight=3, n_estimators=800, reg_alpha=0, reg_lambda=1, subsample=0.6; total time=   6.8s
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=7, min_child_weight=7, n_estimators=800, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   8.1s
[CV] END colsample_bytree=0.6, learning_rate=0.1, max_depth=7, min_child_weight=7, n_estimators=800, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   8.5s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=7, min_child_weight=3, n_estimators=1000, reg_alpha=0.5, reg_lambda=0.5, subsample=0.6; total time=   8.4s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=7,

In [228]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load splits
X_train = pd.read_csv("dataSplits/X_train_encoded.csv")
X_val   = pd.read_csv("dataSplits/X_val_encoded.csv")
X_test  = pd.read_csv("dataSplits/X_test_encoded.csv")

y_train = pd.read_csv("dataSplits/y_train.csv").squeeze()
y_val   = pd.read_csv("dataSplits/y_val.csv").squeeze()
y_test  = pd.read_csv("dataSplits/y_test.csv").squeeze()

# Combine train + val
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])


# Apply log1p transform to targets
y_train_full_log = np.log1p(y_train_full)
y_test_log = np.log1p(y_test)

# Define grid of hyperparameters
param_grid = {
    "max_depth": [ 5, 6, 7],
    "min_child_weight": [1, 3,],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "learning_rate": [0.05],
    "n_estimators": [ 800, 1000],
    "reg_lambda": [ 1, 2],
    "reg_alpha": [0.5, 1]
}

xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)

grid = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="r2",
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"GridSearch starting with {len(grid.param_grid)} parameters, {grid.cv} folds")

# Run grid search on log-transformed targets
grid.fit(X_train_full, y_train_full_log)

print("Best Params:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

# Evaluate on test set

best_model = grid.best_estimator_
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)  # back-transform

print("\nTest Metrics (Best GridSearch Model with Log Transform)")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))


GridSearch starting with 8 parameters, 3 folds
Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 1000, 'reg_alpha': 1, 'reg_lambda': 2, 'subsample': 0.8}
Best CV R²: 0.6923045093102669

Test Metrics (Best GridSearch Model with Log Transform)
MAE: 0.33316781399754275
RMSE: 0.734132260288142
R²: 0.7631504255717959


In [239]:
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import xgboost as xgb


# Get best parameters
best_params = random_search.best_params_
best_params["random_state"] = 42
best_params["n_jobs"] = -1

print("Best Params Selected:", best_params)

# Combine train + val for final training
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])
y_train_full_log = np.log1p(y_train_full)

# Train final model
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_train_full, y_train_full_log)

# Save final model
joblib.dump(final_model, "xgb_final_model.pkl")
print("Final model retrained on full data and saved as xgb_final_model.pkl")


# Evaluate on test set
y_pred_log = final_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nFinal Test Metrics (with Best Params)")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


Best Params Selected: {'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.5, 'n_estimators': 800, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.8, 'random_state': 42, 'n_jobs': -1}
Final model retrained on full data and saved as xgb_final_model.pkl

Final Test Metrics (with Best Params)
MAE:  0.3383
RMSE: 0.7144
R²:   0.7757


## Comparison


We performed two hyperparameter search strategies: **RandomizedSearchCV** and **GridSearchCV**. Below is a comparison of their results:


RandomizedSearchCV Best Hyperparameters
- {'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.5, 'n_estimators': 800,
'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.05,
'colsample_bytree': 0.8}

GridSearchCV Best Hyperparameters
- {'subsample': 0.6, 'reg_lambda': 0.5, 'reg_alpha': 0.5, 'n_estimators': 800,
'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.05,
'colsample_bytree': 0.8, 'random_state': 42, 'n_jobs': -1}

### Comparison of Test Metrics  

| Metric  | RandomizedSearchCV (Best Params) | GridSearchCV (Best Params) |
|---------|----------------------------------|-----------------------------|
| **MAE** | 0.3383                           | 0.3331                      |
| **RMSE**| 0.7144                           | 0.7341                      |
| **R²**  | 0.7757                           | 0.7631                      |



### Conclusion  
Although both search strategies resulted in strong performance, the **best combination of hyperparameters** was found using **RandomizedSearchCV**.  

We retrained the final model using these best parameters and achieved the **best test performance (R² = 0.7757)**.  

Therefore, the **RandomizedSearchCV** combination was selected as the **final model** and saved as `xgb_final_model.pkl`.